In [ ]:
# ==========================================================
# GNSS HYBRID SPOOF DETECTION SYSTEM
# Physics + Deep Learning Architecture
# ==========================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import logging
import warnings

from scipy.linalg import inv
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)

Running on: cpu


In [4]:
# ==========================================================
# LOAD DRONE FLIGHT LOG
# ==========================================================

DATA_PATH = "gps_spoofing_dataset_no_sensor.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,sample_id,timestamp,scenario,gps_lat,gps_lon,gps_alt_m,gps_vel_north_ms,gps_vel_east_ms,gps_vel_down_ms,snr_db_hz,num_satellites,hdop,clock_bias_s,clock_drift_ss,delta_v_ms,delta_p_m,attack_type,label
0,1501,2025-03-04 10:02:30.200,normal,17.392693,78.494689,540.234,5.6032,5.5850,-0.0155,43.13,10,0.944,-3.560000e-09,-7.500000e-12,0.10022,0.2779,none,0
1,2586,2025-03-04 10:04:18.700,simplistic_spoof,17.391970,78.494000,541.400,22.6857,24.5062,-0.0327,46.96,8,0.719,-4.713000e-09,-6.350000e-12,25.41710,318.1394,simplistic_spoof,1
2,2653,2025-03-04 10:04:25.400,meaconing_spoof,17.379584,78.480706,541.400,5.6394,5.7071,0.0384,43.83,9,1.025,1.031706e-06,1.647000e-11,0.13088,1690.5527,meaconing_spoof,1
3,1055,2025-03-04 10:01:45.600,normal,17.390426,78.492306,541.627,5.7588,5.6867,0.0193,41.69,11,0.688,5.131000e-09,2.567000e-11,0.04266,0.4185,none,0
4,705,2025-03-04 10:01:10.600,normal,17.388641,78.490443,541.916,5.6091,5.6435,-0.0098,40.76,10,0.827,2.626000e-09,1.042000e-11,0.10028,0.2722,none,0


In [ ]:
# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df["lat_prev"] = df["latitude"].shift(1)
df["lon_prev"] = df["longitude"].shift(1)

df["distance"] = haversine(
    df["lat_prev"], df["lon_prev"],
    df["latitude"], df["longitude"]
)

df["speed"] = df["distance"]
df["acceleration"] = df["speed"].diff()
df["jerk"] = df["acceleration"].diff()

df["heading_change"] = np.arctan2(
    df["latitude"].diff(),
    df["longitude"].diff()
).diff()

df = df.fillna(0)

df.head()

In [ ]:
# ==========================================================
# EKF RESIDUAL COMPUTATION
# ==========================================================

dt = 1

F = np.array([[1,0,dt,0],
              [0,1,0,dt],
              [0,0,1,0],
              [0,0,0,1]])

H = np.array([[1,0,0,0],
              [0,1,0,0]])

Q = np.eye(4)*0.05
R = np.eye(2)*4

x = np.zeros((4,1))
P = np.eye(4)

residual_scores = []

for i in range(len(df)):
    
    z = np.array([
        [df.iloc[i]["latitude"]],
        [df.iloc[i]["longitude"]]
    ])
    
    x = F @ x
    P = F @ P @ F.T + Q
    
    y = z - H @ x
    S = H @ P @ H.T + R
    
    test_stat = float(y.T @ inv(S) @ y)
    residual_scores.append(test_stat)
    
    K = P @ H.T @ inv(S)
    x = x + K @ y
    P = (np.eye(4) - K @ H) @ P

df["ekf_residual"] = residual_scores

df[["ekf_residual"]].head()

In [ ]:
# ==========================================================
# SIMULATED SPOOF ATTACK WINDOW
# ==========================================================

df["label"] = 0
df.loc[180:210, "label"] = 1

df["label"].value_counts()

In [ ]:
# ==========================================================
# SEQUENCE PREPARATION
# ==========================================================

features = [
    "distance",
    "speed",
    "acceleration",
    "jerk",
    "heading_change",
    "ekf_residual"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

labels = df["label"].values

sequence_length = 25
X_seq, y_seq = [], []

for i in range(len(X_scaled) - sequence_length):
    X_seq.append(X_scaled[i:i+sequence_length])
    y_seq.append(labels[i+sequence_length])

X_seq = torch.tensor(X_seq, dtype=torch.float32).to(DEVICE)
y_seq = torch.tensor(y_seq, dtype=torch.float32).unsqueeze(1).to(DEVICE)

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2
)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

In [ ]:
# ==========================================================
# DEEP TEMPORAL HYBRID MODEL
# ==========================================================

class GNSSHybridModel(nn.Module):

    def __init__(self, input_dim):
        super(GNSSHybridModel, self).__init__()

        self.lstm = nn.LSTM(
            input_dim,
            128,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(256, 1)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 1)

        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        lstm_out, _ = self.lstm(x)

        attn_weights = torch.softmax(
            self.attention(lstm_out), dim=1
        )

        context = torch.sum(attn_weights * lstm_out, dim=1)

        x = torch.relu(self.fc1(context))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x


model = GNSSHybridModel(input_dim=len(features)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:
# ==========================================================
# MODEL TRAINING
# ==========================================================

EPOCHS = 30

for epoch in range(EPOCHS):
    
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

In [ ]:
# ==========================================================
# MODEL EVALUATION
# ==========================================================

model.eval()

with torch.no_grad():
    probs = model(X_test)
    preds = (probs > 0.5).float()

print(classification_report(y_test.cpu(), preds.cpu()))

cm = confusion_matrix(y_test.cpu(), preds.cpu())
print("Confusion Matrix:\n", cm)

fpr, tpr, _ = roc_curve(
    y_test.cpu().numpy(),
    probs.cpu().numpy()
)

roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.title(f"ROC Curve (AUC = {roc_auc:.2f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()